In [2]:
import chromadb
from pathlib import Path

chemin_chroma = Path("../data/chroma_db") 

client = chromadb.PersistentClient(path=str(chemin_chroma))

# Liste les collections existantes
print("Collections :", [c.name for c in client.list_collections()])

Collections : ['fiche']


In [3]:
# Récupère la collection 'fiche'
collection = client.get_collection(name="fiche")
print(f"Nombre total de chunks indexés : {collection.count()}")

Nombre total de chunks indexés : 1038


In [4]:
apercu = collection.peek(limit=3)

for i in range(len(apercu["ids"])):
    print(f"--- ID : {apercu['ids'][i]} ---")
    print("Métadonnées :", apercu["metadatas"][i])
    print("Dimension du vecteur :", len(apercu["embeddings"][i]))
    print("Extrait texte :\n", apercu["documents"][i][:150], "...\n")


--- ID : fiche_1000000000001 ---
Métadonnées : {'marque': 'Kazidomi', 'rayon': 'Céréales et féculents', 'source': 'Open Food Facts', 'code': '1000000000001', 'type': 'fiche_produit', 'nom': 'CHICKPEA - PASTA ORGANIC', 'licence': 'ODbL', 'nutriscore': 'A'}
Dimension du vecteur : 1024
Extrait texte :
 Produit : CHICKPEA - PASTA ORGANIC
Code-barres : 1000000000001
Rayon : Céréales et féculents
Marque : Kazidomi
Nutri-Score : A
Données incomplètes : n ...

--- ID : fiche_10000000000025 ---
Métadonnées : {'licence': 'ODbL', 'marque': 'Kazidomi', 'rayon': 'Céréales et féculents', 'code': '10000000000025', 'nutriscore': 'A', 'nom': 'GNOCCHETTI DE POIS CHICHES BIO', 'type': 'fiche_produit', 'source': 'Open Food Facts'}
Dimension du vecteur : 1024
Extrait texte :
 Produit : GNOCCHETTI DE POIS CHICHES BIO
Code-barres : 10000000000025
Rayon : Céréales et féculents
Marque : Kazidomi
Nutri-Score : A
Données incomplè ...

--- ID : fiche_1000000000003 ---
Métadonnées : {'code': '1000000000003', 'type

### Fiches :

In [5]:
docs_off = collection.get(where={"type": "fiche_produit"})
print("Nombre de fiches indexés :", len(docs_off["ids"]))

Nombre de fiches indexés : 1000


In [6]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client_openai = OpenAI(
    base_url=os.environ.get("LLM_BASE_URL", "http://localhost:11434/v1"),
    api_key=os.environ.get("LLM_API_KEY", "ollama"),
)
modele_embed = os.environ.get("LLM_EMBEDDINGS", "qwen3-embedding:0.6b")

# Vectorisation de la requête
requete = "biscuits au chocolat riches en fibres"
rep = client_openai.embeddings.create(model=modele_embed, input=[requete])
vecteur_requete = rep.data[0].embedding

# Recherche dans ChromaDB
resultats = collection.query(
    query_embeddings=[vecteur_requete],
    n_results=3,
    where={"type": "fiche_produit"}
)

for doc_id, dist, meta, doc in zip(
    resultats["ids"][0],
    resultats["distances"][0],
    resultats["metadatas"][0],
    resultats["documents"][0]
):
    # plus le score est proche de 0, plus le produit correspond à la recherche.
    print(f"Score (distance cosinus) : {dist:.4f} | ID : {doc_id}")
    print(f"Produit : {meta.get('nom')} (Nutri-Score : {meta.get('nutriscore', 'N/A')})")
    print("-" * 50)

Score (distance cosinus) : 0.3629 | ID : fiche_1001003803
Produit : BISCUITS AUX FRUITS (Nutri-Score : D)
--------------------------------------------------
Score (distance cosinus) : 0.3642 | ID : fiche_10992937
Produit : BISCUIT CHOCO NOIR INTENSE (Nutri-Score : E)
--------------------------------------------------
Score (distance cosinus) : 0.3757 | ID : fiche_10130713
Produit : TOO GOOD BISCUITS (Nutri-Score : D)
--------------------------------------------------


In [7]:
# Récupère les métadonnées de toutes les fiches pour lister les rayons uniques
donnees = collection.get(where={"type": "fiche_produit"}, include=["metadatas"])
rayons_dispos = sorted({m["rayon"] for m in donnees["metadatas"] if "rayon" in m})

print("Rayons disponibles :", rayons_dispos)

Rayons disponibles : ['Boissons', 'Céréales et féculents', 'Produits laitiers', 'Snacks sucrés', 'Viandes, poissons et œufs']


In [8]:
rayon_cible = "Snacks sucrés"

# Vectorisation de la requête
requete = "biscuits au chocolat"
rep = client_openai.embeddings.create(model=modele_embed, input=[requete])
vecteur_requete = rep.data[0].embedding

# Recherche avec double filtre dans ChromaDB
resultats = collection.query(
    query_embeddings=[vecteur_requete],
    n_results=3,
    where={
        "$and": [
            {"type": {"$eq": "fiche_produit"}},
            {"rayon": {"$eq": rayon_cible}}
        ]
    }
)

for doc_id, dist, meta in zip(
    resultats["ids"][0],
    resultats["distances"][0],
    resultats["metadatas"][0]
):
    print(f"Distance : {dist:.4f} | Rayon : {meta.get('rayon')}")
    print(f"Produit : {meta.get('nom')} (Nutri-Score : {meta.get('nutriscore', 'N/A')})")
    print("-" * 50)

Distance : 0.4247 | Rayon : Snacks sucrés
Produit : VÉRITABLE PÂTE À TARTINER NOISETTES CHOCOLAT NOIR (Nutri-Score : E)
--------------------------------------------------
Distance : 0.4254 | Rayon : Snacks sucrés
Produit : BISCUIT CHOCO NOIR INTENSE (Nutri-Score : E)
--------------------------------------------------
Distance : 0.4285 | Rayon : Snacks sucrés
Produit : CHOCOLAT NOIR BROWNIE (Nutri-Score : E)
--------------------------------------------------


### Documents :

In [9]:
docs_off = collection.get(where={"type": "document_officiel"})
print("Nombre de documents officiels indexés :", len(docs_off["ids"]))
print(collection.get(where={"type": "document_officiel"}, include=["metadatas"])["ids"])

Nombre de documents officiels indexés : 38
['doc_viepub-ns_chunk_0', 'doc_santefr-ns_chunk_0', 'doc_santefr-ns_chunk_1', 'doc_santefr-ns_chunk_2', 'doc_santefr-ns_chunk_3', 'doc_santefr-ns_chunk_4', 'doc_santefr-ns_chunk_5', 'doc_santefr-ns_chunk_6', 'doc_santefr-ns_chunk_7', 'doc_spf-ns_chunk_0', 'doc_spf-ns_chunk_1', 'doc_spf-ns_chunk_2', 'doc_spf-ns_chunk_3', 'doc_spf-ns_chunk_4', 'doc_spf-ns_chunk_5', 'doc_spf-ns_chunk_6', 'doc_spf-ns_chunk_7', 'doc_spf-ns_chunk_8', 'doc_spf-ns_chunk_9', 'doc_spf-ns_chunk_10', 'doc_spf-ns_chunk_11', 'doc_spf-ns_chunk_12', 'doc_spf-ns_chunk_13', 'doc_spf-ns_chunk_14', 'doc_spf-ns_chunk_15', 'doc_spf-ns_chunk_16', 'doc_spf-ns_chunk_17', 'doc_spf-ns_chunk_18', 'doc_spf-ns_chunk_19', 'doc_spf-ns_chunk_20', 'doc_spf-ns_chunk_21', 'doc_spf-ns_chunk_22', 'doc_spf-ns_chunk_23', 'doc_spf-ns_chunk_24', 'doc_pnns-adultes_chunk_0', 'doc_pnns-adultes_chunk_1', 'doc_pnns-adultes_chunk_2', 'doc_pnns-adultes_chunk_3']


In [10]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client_openai = OpenAI(
    base_url=os.environ.get("LLM_BASE_URL", "http://localhost:11434/v1"),
    api_key=os.environ.get("LLM_API_KEY", "ollama"),
)
modele_embed = os.environ.get("LLM_EMBEDDINGS", "qwen3-embedding:0.6b")

# Vectorisation de la requête
requete = "comment est calculé le Nutri-Score ?"
rep = client_openai.embeddings.create(model=modele_embed, input=[requete])
vecteur_requete = rep.data[0].embedding

# Recherche dans ChromaDB
resultats = collection.query(
    query_embeddings=[vecteur_requete],
    n_results=3,
    where={"type": "document_officiel"}
)

for doc_id, dist, meta, doc in zip(
    resultats["ids"][0],
    resultats["distances"][0],
    resultats["metadatas"][0],
    resultats["documents"][0]
):
    # plus le score est proche de 0, plus le produit correspond à la recherche.
    print(f"Score (distance cosinus) : {dist:.4f} | ID : {doc_id}")
    print(f"Source : {meta.get('source', 'N/A')} | URL : {meta.get('url', 'N/A')}")
    print(f"Extrait : {doc[:150]}...")
    print("-" * 50)

Score (distance cosinus) : 0.1996 | ID : doc_spf-ns_chunk_5
Source : Nutri-Score — Santé publique France | URL : https://www.santepubliquefrance.fr/determinants-de-sante/nutrition-et-activite-physique/articles/nutri-score
Extrait : rrêté fixant les nouvelles modalités de calcul.
L'ensemble des articles scientifiques et documents publiés relatifs au Nutri-Score est accessible ici....
--------------------------------------------------
Score (distance cosinus) : 0.2238 | ID : doc_spf-ns_chunk_11
Source : Nutri-Score — Santé publique France | URL : https://www.santepubliquefrance.fr/determinants-de-sante/nutrition-et-activite-physique/articles/nutri-score
Extrait : /
version anglaise
), qui permettent le calcul du Nutri-Score de vos produits (ce fichier n'est pas à transmettre à l'Oqali).
Pour tout renseignement
...
--------------------------------------------------
Score (distance cosinus) : 0.2321 | ID : doc_santefr-ns_chunk_2
Source : Ce qu'il faut savoir sur le nouveau Nutri-Score — Sa

In [12]:
import chromadb
col = chromadb.PersistentClient(path=str(chemin_chroma)).get_collection("fiche")
res_pnns = col.get(where={"type": "document_officiel"})
pnns_ids = [i for i in res_pnns["ids"] if "pnns-adultes" in i]
print(f"Chunks pnns-adultes indexés : {len(pnns_ids)}")
if pnns_ids:
    print("Aperçu chunk 0 :", res_pnns["documents"][res_pnns["ids"].index(pnns_ids[0])][:300])

Chunks pnns-adultes indexés : 4
Aperçu chunk 0 : Mois sans tabac 2026 : des outils, du soutien, pour réussir le pari d’une vie sans tabac
Santé publique France, le ministère en charge de la Santé et l’Assurance Maladie unissent leurs forces pour offrir un accompagnement gratuit et personnalisé à tous ceux qui souhaitent arrêter de fumer
Participer
